In [2]:
# 🔎 Inference + Metrics Export (ConvNeXt checkpoint) — publish-ready
import os, cv2, json, torch, timm
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from timm.data import resolve_model_data_config

# ===== PATHS =====
TEST_DIR = "/home/gpu/PSL-Cropped/train" 
CKPT     = "/home/gpu/PSL Isolated Signs Datasets/VIT-TINY-UA+PSL-Images-Train-CHKPT/best_Deit_Tiny_psl.pth"
OUT_DIR  = "/home/gpu/PSL Isolated Signs Datasets/Resutls-VIT-Tiny-CFRT-Images-test-23-10-25"
os.makedirs(OUT_DIR, exist_ok=True)

# ===== UTILS =====
def safe_imread_rgb(path: str):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise RuntimeError(f"Failed to load image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

class AlbumentationsImageFolder(ImageFolder):
    def __init__(self, root, transform):
        super().__init__(root=root); self.atransform = transform
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        img = safe_imread_rgb(path)
        img = self.atransform(image=img)["image"]
        return img, target

def make_val_tfms(img_size, mean, std):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
        A.Normalize(mean=tuple(mean), std=tuple(std)),
        ToTensorV2(),
    ])

@torch.no_grad()
def evaluate(model, loader, device, num_classes):
    crit = torch.nn.CrossEntropyLoss()
    y_true, y_pred, total, loss_meter, top5_correct = [], [], 0, 0.0, 0
    k = min(5, num_classes)

    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        logits = model(images)
        loss = crit(logits, targets)

        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        y_true.extend(targets.cpu().numpy()); y_pred.extend(preds.cpu().numpy())
        loss_meter += loss.item() * images.size(0); total += images.size(0)

        topk_idx = torch.topk(probs, k=k, dim=1).indices
        top5_correct += topk_idx.eq(targets.view(-1,1)).any(dim=1).sum().item()

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm   = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    report = classification_report(y_true, y_pred, labels=list(range(num_classes)), output_dict=True)

    return {
        "acc": acc*100, "prec": prec*100, "rec": rec*100, "f1": f1*100,
        "top1": acc*100, "top5": 100*top5_correct/max(1,total),
        "loss": loss_meter/max(1,total), "cm": cm,
        "report": report
    }

# ===== MAIN =====
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(CKPT, map_location=device)

# Pull model metadata from checkpoint; keep robust fallbacks
model_name  = ckpt.get("model_name", "convnext_base")  # fallback if missing
num_classes = int(ckpt.get("classes", 1000))
class_names = ckpt.get("class_names", [str(i) for i in range(num_classes)])

# Build model and load weights
model = timm.create_model(model_name, pretrained=False, num_classes=num_classes).to(device)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()

# Resolve transforms: prefer saved cfg/img_size, else derive from model
cfg = ckpt.get("cfg", None)
if cfg is None:
    cfg = resolve_model_data_config(model)
img_size = ckpt.get("img_size", cfg["input_size"][-1] if "input_size" in cfg else 224)
mean = cfg.get("mean", (0.5, 0.5, 0.5))
std  = cfg.get("std",  (0.5, 0.5, 0.5))

# Dataset / loader
tfms = make_val_tfms(img_size, mean, std)
test_ds = AlbumentationsImageFolder(TEST_DIR, tfms)

# Ensure class order matches (publishable CM/per-class labels)
assert test_ds.classes == class_names, "Class mismatch with checkpoint class_names; ensure identical folder order."
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

# Evaluate
metrics = evaluate(model, test_loader, device, num_classes)

# ===== SAVE RESULTS =====
# 1) Human-readable TXT
with open(os.path.join(OUT_DIR, "metrics.txt"), "w") as f:
    f.write(f"Model: {model_name}\n")
    f.write(f"Checkpoint: {os.path.basename(CKPT)}\n")
    f.write(f"Test Images: {len(test_ds)} | Classes: {num_classes}\n\n")
    f.write(f"Top-1 Acc: {metrics['top1']:.2f}%\n")
    f.write(f"Top-5 Acc: {metrics['top5']:.2f}%\n")
    f.write(f"Precision (macro): {metrics['prec']:.2f}%\n")
    f.write(f"Recall (macro): {metrics['rec']:.2f}%\n")
    f.write(f"F1 (macro): {metrics['f1']:.2f}%\n")
    f.write(f"CrossEntropy Loss: {metrics['loss']:.6f}\n")

# 2) JSON (with numpy-friendly conversion)
def _to_jsonable(x):
    import numpy as _np
    if isinstance(x, _np.ndarray): return x.tolist()
    return x
json.dump(
    {**{"model": model_name, "checkpoint": os.path.basename(CKPT),
        "num_test_images": len(test_ds), "num_classes": num_classes, "class_names": class_names},
     **metrics},
    open(os.path.join(OUT_DIR, "metrics.json"), "w"), indent=2, default=_to_jsonable
)

# 3) CSVs
pd.DataFrame(metrics["cm"], index=class_names, columns=class_names)\
  .to_csv(os.path.join(OUT_DIR,"confusion_matrix.csv"))
rows=[]
for i, cls in enumerate(class_names):
    stats = metrics["report"].get(str(i), {"precision":0,"recall":0,"f1-score":0,"support":0})
    rows.append({"class": cls, "precision": stats.get("precision",0.0),
                 "recall": stats.get("recall",0.0),
                 "f1": stats.get("f1-score",0.0),
                 "support": stats.get("support",0)})
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR,"per_class.csv"), index=False)

# 4) Excel (summary + CM + per-class)
with pd.ExcelWriter(os.path.join(OUT_DIR,"metrics.xlsx"), engine="openpyxl") as writer:
    pd.DataFrame([{
        "model": model_name,
        "checkpoint": os.path.basename(CKPT),
        "test_images": len(test_ds),
        "classes": num_classes,
        "Top1_%": metrics["top1"],
        "Top5_%": metrics["top5"],
        "Macro_Precision_%": metrics["prec"],
        "Macro_Recall_%": metrics["rec"],
        "Macro_F1_%": metrics["f1"],
        "Loss": metrics["loss"],
    }]).to_excel(writer, index=False, sheet_name="summary")
    pd.DataFrame(metrics["cm"], index=class_names, columns=class_names)\
        .to_excel(writer, sheet_name="confusion_matrix")
    pd.DataFrame(rows).to_excel(writer, index=False, sheet_name="per_class")

print(f"✅ ConvNeXt results saved to: {OUT_DIR}")


/tmp/ipykernel_72880/3402011528.py:80: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CKPT, map_location=device)


✅ ConvNeXt results saved to: /home/gpu/PSL Isolated Signs Datasets/Resutls-VIT-Tiny-CFRT-Images-test-23-10-25


/home/gpu/.conda/envs/python305/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/gpu/.conda/envs/python305/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/gpu/.conda/envs/python305/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

In [3]:
# 🔎 Inference + Metrics Export (ConvNeXt checkpoint) — publish-ready
import os, cv2, json, torch, timm
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from timm.data import resolve_model_data_config

# ===== PATHS =====
TEST_DIR = "/home/gpu/PSL Isolated Signs Datasets/PSL-Dataset-2021/train" 
CKPT     = "/home/gpu/PSL Isolated Signs Datasets/VIT-TINY-UA+CFRT-Train-CHKPT/best_Deit_Tiny_psl.pth"
OUT_DIR  = "/home/gpu/PSL Isolated Signs Datasets/Resutls-VIT-Tiny-PSL-Images-test-23-10-25"
os.makedirs(OUT_DIR, exist_ok=True)

# ===== UTILS =====
def safe_imread_rgb(path: str):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise RuntimeError(f"Failed to load image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

class AlbumentationsImageFolder(ImageFolder):
    def __init__(self, root, transform):
        super().__init__(root=root); self.atransform = transform
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        img = safe_imread_rgb(path)
        img = self.atransform(image=img)["image"]
        return img, target

def make_val_tfms(img_size, mean, std):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
        A.Normalize(mean=tuple(mean), std=tuple(std)),
        ToTensorV2(),
    ])

@torch.no_grad()
def evaluate(model, loader, device, num_classes):
    crit = torch.nn.CrossEntropyLoss()
    y_true, y_pred, total, loss_meter, top5_correct = [], [], 0, 0.0, 0
    k = min(5, num_classes)

    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        logits = model(images)
        loss = crit(logits, targets)

        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        y_true.extend(targets.cpu().numpy()); y_pred.extend(preds.cpu().numpy())
        loss_meter += loss.item() * images.size(0); total += images.size(0)

        topk_idx = torch.topk(probs, k=k, dim=1).indices
        top5_correct += topk_idx.eq(targets.view(-1,1)).any(dim=1).sum().item()

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm   = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    report = classification_report(y_true, y_pred, labels=list(range(num_classes)), output_dict=True)

    return {
        "acc": acc*100, "prec": prec*100, "rec": rec*100, "f1": f1*100,
        "top1": acc*100, "top5": 100*top5_correct/max(1,total),
        "loss": loss_meter/max(1,total), "cm": cm,
        "report": report
    }

# ===== MAIN =====
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(CKPT, map_location=device)

# Pull model metadata from checkpoint; keep robust fallbacks
model_name  = ckpt.get("model_name", "convnext_base")  # fallback if missing
num_classes = int(ckpt.get("classes", 1000))
class_names = ckpt.get("class_names", [str(i) for i in range(num_classes)])

# Build model and load weights
model = timm.create_model(model_name, pretrained=False, num_classes=num_classes).to(device)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()

# Resolve transforms: prefer saved cfg/img_size, else derive from model
cfg = ckpt.get("cfg", None)
if cfg is None:
    cfg = resolve_model_data_config(model)
img_size = ckpt.get("img_size", cfg["input_size"][-1] if "input_size" in cfg else 224)
mean = cfg.get("mean", (0.5, 0.5, 0.5))
std  = cfg.get("std",  (0.5, 0.5, 0.5))

# Dataset / loader
tfms = make_val_tfms(img_size, mean, std)
test_ds = AlbumentationsImageFolder(TEST_DIR, tfms)

# Ensure class order matches (publishable CM/per-class labels)
assert test_ds.classes == class_names, "Class mismatch with checkpoint class_names; ensure identical folder order."
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

# Evaluate
metrics = evaluate(model, test_loader, device, num_classes)

# ===== SAVE RESULTS =====
# 1) Human-readable TXT
with open(os.path.join(OUT_DIR, "metrics.txt"), "w") as f:
    f.write(f"Model: {model_name}\n")
    f.write(f"Checkpoint: {os.path.basename(CKPT)}\n")
    f.write(f"Test Images: {len(test_ds)} | Classes: {num_classes}\n\n")
    f.write(f"Top-1 Acc: {metrics['top1']:.2f}%\n")
    f.write(f"Top-5 Acc: {metrics['top5']:.2f}%\n")
    f.write(f"Precision (macro): {metrics['prec']:.2f}%\n")
    f.write(f"Recall (macro): {metrics['rec']:.2f}%\n")
    f.write(f"F1 (macro): {metrics['f1']:.2f}%\n")
    f.write(f"CrossEntropy Loss: {metrics['loss']:.6f}\n")

# 2) JSON (with numpy-friendly conversion)
def _to_jsonable(x):
    import numpy as _np
    if isinstance(x, _np.ndarray): return x.tolist()
    return x
json.dump(
    {**{"model": model_name, "checkpoint": os.path.basename(CKPT),
        "num_test_images": len(test_ds), "num_classes": num_classes, "class_names": class_names},
     **metrics},
    open(os.path.join(OUT_DIR, "metrics.json"), "w"), indent=2, default=_to_jsonable
)

# 3) CSVs
pd.DataFrame(metrics["cm"], index=class_names, columns=class_names)\
  .to_csv(os.path.join(OUT_DIR,"confusion_matrix.csv"))
rows=[]
for i, cls in enumerate(class_names):
    stats = metrics["report"].get(str(i), {"precision":0,"recall":0,"f1-score":0,"support":0})
    rows.append({"class": cls, "precision": stats.get("precision",0.0),
                 "recall": stats.get("recall",0.0),
                 "f1": stats.get("f1-score",0.0),
                 "support": stats.get("support",0)})
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR,"per_class.csv"), index=False)

# 4) Excel (summary + CM + per-class)
with pd.ExcelWriter(os.path.join(OUT_DIR,"metrics.xlsx"), engine="openpyxl") as writer:
    pd.DataFrame([{
        "model": model_name,
        "checkpoint": os.path.basename(CKPT),
        "test_images": len(test_ds),
        "classes": num_classes,
        "Top1_%": metrics["top1"],
        "Top5_%": metrics["top5"],
        "Macro_Precision_%": metrics["prec"],
        "Macro_Recall_%": metrics["rec"],
        "Macro_F1_%": metrics["f1"],
        "Loss": metrics["loss"],
    }]).to_excel(writer, index=False, sheet_name="summary")
    pd.DataFrame(metrics["cm"], index=class_names, columns=class_names)\
        .to_excel(writer, sheet_name="confusion_matrix")
    pd.DataFrame(rows).to_excel(writer, index=False, sheet_name="per_class")

print(f"✅ ConvNeXt results saved to: {OUT_DIR}")


/tmp/ipykernel_72880/985429849.py:80: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CKPT, map_location=device)


✅ ConvNeXt results saved to: /home/gpu/PSL Isolated Signs Datasets/Resutls-VIT-Tiny-PSL-Images-test-23-10-25


/home/gpu/.conda/envs/python305/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/gpu/.conda/envs/python305/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/gpu/.conda/envs/python305/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

In [4]:
# 🔎 Inference + Metrics Export (ConvNeXt checkpoint) — publish-ready
import os, cv2, json, torch, timm
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from timm.data import resolve_model_data_config

# ===== PATHS =====
TEST_DIR = "/home/gpu/PSL Isolated Signs Datasets/splits-of-combined-cropped-dataset-18-8/test"
CKPT     = "/home/gpu/PSL Isolated Signs Datasets/VIT-Tiny-stratified-train-checkpoint/best_Deit_Tiny_psl.pth"
OUT_DIR  = "/home/gpu/PSL Isolated Signs Datasets/Resutls-VIT-Tiny-stratified-23-10-25"
os.makedirs(OUT_DIR, exist_ok=True)

# ===== UTILS =====
def safe_imread_rgb(path: str):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise RuntimeError(f"Failed to load image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

class AlbumentationsImageFolder(ImageFolder):
    def __init__(self, root, transform):
        super().__init__(root=root); self.atransform = transform
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        img = safe_imread_rgb(path)
        img = self.atransform(image=img)["image"]
        return img, target

def make_val_tfms(img_size, mean, std):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
        A.Normalize(mean=tuple(mean), std=tuple(std)),
        ToTensorV2(),
    ])

@torch.no_grad()
def evaluate(model, loader, device, num_classes):
    crit = torch.nn.CrossEntropyLoss()
    y_true, y_pred, total, loss_meter, top5_correct = [], [], 0, 0.0, 0
    k = min(5, num_classes)

    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        logits = model(images)
        loss = crit(logits, targets)

        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        y_true.extend(targets.cpu().numpy()); y_pred.extend(preds.cpu().numpy())
        loss_meter += loss.item() * images.size(0); total += images.size(0)

        topk_idx = torch.topk(probs, k=k, dim=1).indices
        top5_correct += topk_idx.eq(targets.view(-1,1)).any(dim=1).sum().item()

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm   = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    report = classification_report(y_true, y_pred, labels=list(range(num_classes)), output_dict=True)

    return {
        "acc": acc*100, "prec": prec*100, "rec": rec*100, "f1": f1*100,
        "top1": acc*100, "top5": 100*top5_correct/max(1,total),
        "loss": loss_meter/max(1,total), "cm": cm,
        "report": report
    }

# ===== MAIN =====
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(CKPT, map_location=device)

# Pull model metadata from checkpoint; keep robust fallbacks
model_name  = ckpt.get("model_name", "convnext_base")  # fallback if missing
num_classes = int(ckpt.get("classes", 1000))
class_names = ckpt.get("class_names", [str(i) for i in range(num_classes)])

# Build model and load weights
model = timm.create_model(model_name, pretrained=False, num_classes=num_classes).to(device)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()

# Resolve transforms: prefer saved cfg/img_size, else derive from model
cfg = ckpt.get("cfg", None)
if cfg is None:
    cfg = resolve_model_data_config(model)
img_size = ckpt.get("img_size", cfg["input_size"][-1] if "input_size" in cfg else 224)
mean = cfg.get("mean", (0.5, 0.5, 0.5))
std  = cfg.get("std",  (0.5, 0.5, 0.5))

# Dataset / loader
tfms = make_val_tfms(img_size, mean, std)
test_ds = AlbumentationsImageFolder(TEST_DIR, tfms)

# Ensure class order matches (publishable CM/per-class labels)
assert test_ds.classes == class_names, "Class mismatch with checkpoint class_names; ensure identical folder order."
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

# Evaluate
metrics = evaluate(model, test_loader, device, num_classes)

# ===== SAVE RESULTS =====
# 1) Human-readable TXT
with open(os.path.join(OUT_DIR, "metrics.txt"), "w") as f:
    f.write(f"Model: {model_name}\n")
    f.write(f"Checkpoint: {os.path.basename(CKPT)}\n")
    f.write(f"Test Images: {len(test_ds)} | Classes: {num_classes}\n\n")
    f.write(f"Top-1 Acc: {metrics['top1']:.2f}%\n")
    f.write(f"Top-5 Acc: {metrics['top5']:.2f}%\n")
    f.write(f"Precision (macro): {metrics['prec']:.2f}%\n")
    f.write(f"Recall (macro): {metrics['rec']:.2f}%\n")
    f.write(f"F1 (macro): {metrics['f1']:.2f}%\n")
    f.write(f"CrossEntropy Loss: {metrics['loss']:.6f}\n")

# 2) JSON (with numpy-friendly conversion)
def _to_jsonable(x):
    import numpy as _np
    if isinstance(x, _np.ndarray): return x.tolist()
    return x
json.dump(
    {**{"model": model_name, "checkpoint": os.path.basename(CKPT),
        "num_test_images": len(test_ds), "num_classes": num_classes, "class_names": class_names},
     **metrics},
    open(os.path.join(OUT_DIR, "metrics.json"), "w"), indent=2, default=_to_jsonable
)

# 3) CSVs
pd.DataFrame(metrics["cm"], index=class_names, columns=class_names)\
  .to_csv(os.path.join(OUT_DIR,"confusion_matrix.csv"))
rows=[]
for i, cls in enumerate(class_names):
    stats = metrics["report"].get(str(i), {"precision":0,"recall":0,"f1-score":0,"support":0})
    rows.append({"class": cls, "precision": stats.get("precision",0.0),
                 "recall": stats.get("recall",0.0),
                 "f1": stats.get("f1-score",0.0),
                 "support": stats.get("support",0)})
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR,"per_class.csv"), index=False)

# 4) Excel (summary + CM + per-class)
with pd.ExcelWriter(os.path.join(OUT_DIR,"metrics.xlsx"), engine="openpyxl") as writer:
    pd.DataFrame([{
        "model": model_name,
        "checkpoint": os.path.basename(CKPT),
        "test_images": len(test_ds),
        "classes": num_classes,
        "Top1_%": metrics["top1"],
        "Top5_%": metrics["top5"],
        "Macro_Precision_%": metrics["prec"],
        "Macro_Recall_%": metrics["rec"],
        "Macro_F1_%": metrics["f1"],
        "Loss": metrics["loss"],
    }]).to_excel(writer, index=False, sheet_name="summary")
    pd.DataFrame(metrics["cm"], index=class_names, columns=class_names)\
        .to_excel(writer, sheet_name="confusion_matrix")
    pd.DataFrame(rows).to_excel(writer, index=False, sheet_name="per_class")

print(f"✅ ConvNeXt results saved to: {OUT_DIR}")


/tmp/ipykernel_72880/1624068972.py:80: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CKPT, map_location=device)


✅ ConvNeXt results saved to: /home/gpu/PSL Isolated Signs Datasets/Resutls-VIT-Tiny-stratified-23-10-25


In [4]:
# === No-TTA inference matching your ConvNeXt training pipeline ===
# deps: albumentations==2.x, timm, torch, torchvision, opencv-python

import os
from pathlib import Path
import numpy as np
import torch, timm
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ------------------ CONFIG ------------------
WEIGHTS = "/home/gpu/PSL Isolated Signs Datasets/ckpts_convnext_jupyter/best.pth"
DATA_ROOT = "/home/gpu/PSL Isolated Signs Datasets/Combined-cropped-split/test"  # your eval folder
OUT_DIR   = "/home/gpu/PSL Isolated Signs Datasets/eval_no_tta_out-22"
MISSING_CLASS_NAME = "ں"   # keep empty "" if not needed
print("starting the inference on validation datset...")
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------ UTILITIES ------------------
def ensure_missing_class_folder(root: str, class_name: str):
    if not class_name:
        return
    target = Path(root) / class_name
    if not target.exists():
        target.mkdir(parents=True, exist_ok=True)
        print(f"[FIX] Created empty class folder: {target}")

def safe_imread_rgb(path: str):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise RuntimeError(f"Failed to read: {path}")
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.ndim == 3:
        if img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def _cv2_interp_from_timm(interp: str):
    lut = {
        'bilinear': cv2.INTER_LINEAR,
        'bicubic':  cv2.INTER_CUBIC,
        'area':     cv2.INTER_AREA,
        'nearest':  cv2.INTER_NEAREST,
        'lanczos':  cv2.INTER_LANCZOS4,
        'linear':   cv2.INTER_LINEAR,
        'cubic':    cv2.INTER_CUBIC,
    }
    return lut.get(str(interp).lower(), cv2.INTER_CUBIC)

def make_val_tfms(img_size, mean, std, interp):
    cv2_interp = _cv2_interp_from_timm(interp)
    return A.Compose([
        A.LongestMaxSize(max_size=img_size, interpolation=cv2_interp),
        A.PadIfNeeded(min_height=img_size, min_width=img_size,
                      border_mode=cv2.BORDER_REFLECT_101),
        A.Normalize(mean=tuple(mean), std=tuple(std)),
        ToTensorV2(),
    ])

# ------------------ EVAL (no TTA, label-mapping fallback) ------------------
@torch.no_grad()
def eval_no_tta(data_root: str, weights: str, out_dir: str, bs=512, workers=8):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 1) Load checkpoint metadata
    ckpt = torch.load(weights, map_location="cpu")
    model_name = ckpt.get("model_name", "convnext_base.fb_in22k_ft_in1k")
    n_cls_ckpt = int(ckpt.get("classes", 39))

    # Prefer the saved timm data config from training
    cfg = ckpt.get("cfg", {})
    # img_size: fall back to cfg['input_size'][-1] or 224
    img_size = int(ckpt.get("img_size", cfg.get("input_size", [3, 224, 224])[-1]))
    mean = tuple(cfg.get("mean", (0.5, 0.5, 0.5)))
    std  = tuple(cfg.get("std",  (0.5, 0.5, 0.5)))
    interp = cfg.get("interpolation", "bicubic")
    state = ckpt["model"]

    # 2) Build model & load weights
    model = timm.create_model(model_name, pretrained=False, num_classes=n_cls_ckpt)
    # robust to minor naming/version diffs
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        print("[WARN] load_state_dict mismatches:",
              f"missing={len(missing)} unexpected={len(unexpected)}")
    model.to(device).eval()

    # 3) Ensure class index alignment if a class folder is missing
    ensure_missing_class_folder(data_root, MISSING_CLASS_NAME)

    # 4) Dataset with the SAME val transform as training
    val_tfm = make_val_tfms(img_size, mean, std, interp)

    class ImageFolderVal(ImageFolder):
        def __init__(self, root): super().__init__(root=root)
        def __getitem__(self, idx):
            path, y = self.samples[idx]
            img = safe_imread_rgb(path)
            x = val_tfm(image=img)["image"]
            return x, y

    ds = ImageFolderVal(data_root)
    n_cls_ds = len(ds.classes)
    N = len(ds)
    print(f"[INFO] model={model_name} | classes_ckpt={n_cls_ckpt} | classes_ds={n_cls_ds} | N={N} | img={img_size} | mean={mean} | std={std} | interp={interp}")

    dl = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=workers, pin_memory=True)

    # 5) Single deterministic pass (no TTA)
    logits_all = torch.zeros((N, n_cls_ckpt), dtype=torch.float32, device=device)
    ofs = 0
    use_amp = torch.cuda.is_available()
    for xb, yb in dl:
        xb = xb.to(device, non_blocking=True)
        if use_amp:
            with torch.amp.autocast("cuda"):
                logits = model(xb)
        else:
            logits = model(xb)
        B = xb.size(0)
        logits_all[ofs:ofs+B] = logits
        ofs += B

    preds = logits_all.argmax(1).cpu().numpy()
    targets = np.array([y for _, y in ds.samples], dtype=np.int64)

    # 6) Handle class count mismatch via optimal mapping (rare if folder fix is done)
    if n_cls_ds != n_cls_ckpt:
        print("[WARN] class count mismatch; applying optimal label mapping.")
        cm_rect = np.zeros((n_cls_ds, n_cls_ckpt), dtype=np.int64)
        for t, p in zip(targets, preds):
            cm_rect[t, p] += 1
        try:
            from scipy.optimize import linear_sum_assignment
            row_ind, col_ind = linear_sum_assignment(cm_rect.max() - cm_rect)
        except Exception:
            # greedy fallback
            row_ind, col_ind = [], []
            used = set()
            for r in range(n_cls_ds):
                order = np.argsort(-cm_rect[r])
                c_pick = next((c for c in order if c not in used), int(order[0]))
                row_ind.append(r); col_ind.append(c_pick); used.add(c_pick)
            row_ind, col_ind = np.array(row_ind), np.array(col_ind)

        inv_map = {int(c): int(r) for r, c in zip(row_ind, col_ind)}
        preds_mapped = np.array([inv_map.get(int(p), -1) for p in preds], dtype=np.int64)
        mask = preds_mapped >= 0
        acc_raw = (preds == targets).mean() * 100.0
        acc_mapped = (preds_mapped[mask] == targets[mask]).mean() * 100.0 if mask.any() else 0.0
        with open(os.path.join(out_dir, "CONVNEXT-NO-TTA-summary-mapped.txt"), "w", encoding="utf-8") as f:
            f.write(f"Raw acc: {acc_raw:.2f}%\nMapped acc: {acc_mapped:.2f}%\n")
        print(f"Raw acc: {acc_raw:.2f}% | Mapped acc: {acc_mapped:.2f}% → {out_dir}")
        return acc_mapped

    # 7) Normal accuracy (aligned classes)
    acc = (preds == targets).mean() * 100.0
    with open(os.path.join(out_dir, "CONVNEXT-NO-TTA-summary.txt"), "w", encoding="utf-8") as f:
        f.write(f"Accuracy: {acc:.2f}%\n")
        f.write(f"model={model_name} | img={img_size} | mean={mean} | std={std} | interp={interp}\n")
    print(f"Aligned classes. Acc: {acc:.2f}% → {out_dir}")
    return acc

# ------------------ RUN ------------------
if __name__ == "__main__":
    acc = eval_no_tta(
        data_root=DATA_ROOT,
        weights=WEIGHTS,
        out_dir=OUT_DIR,
        bs=50,        # you can lower if VRAM limits
        workers=20
    )
    print("Final Accuracy:", f"{acc:.2f}%")


starting the inference on validation datset...


/tmp/ipykernel_3438/1675408328.py:74: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(weights, map_location="cpu")


[INFO] model=convnextv2_large | classes_ckpt=39 | classes_ds=39 | N=5249 | img=224 | mean=(0.5, 0.5, 0.5) | std=(0.5, 0.5, 0.5) | interp=bicubic
Aligned classes. Acc: 11.79% → /home/gpu/PSL Isolated Signs Datasets/eval_no_tta_out-22
Final Accuracy: 11.79%
